# 桶排序
对MA因子进行传统的桶排序，并且构建对冲组合，分析每个桶的收益和夏普。 

不同的任务都可以使用该脚本  

异质性分析数据需要预先筛选因子数据  




## 导入库

In [135]:
import warnings
from typing import Any
from pathlib import Path
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as sp
import plotly.io as pio
import plotly.graph_objects as go
import plotly.subplots as sp
import numpy as np
import statsmodels.api as sm
from scipy.stats import t as t_dist

import os
import dotenv
dotenv.load_dotenv()

True

## 超参数 

In [ ]:
CONNECTION_URL = os.getenv("POSTGRES_URL")
ENGINE = "adbc"

TASK_ID_PREFIX = 'baseline1'  # 任务id前缀
RESULTS_BASE_DIR = '/home/frank/files/programs/GraduationThesis/result' # 基本数据路径
SAVE_BASE_DIR = f'/home/frank/files/programs/GraduationThesis/empirical/{TASK_ID_PREFIX}' # 保存基本路径
SAVE = True # 是否保存数据

BUCKETS_NUM = 10 # 分桶数  
HETER = False # 如果不使用异质性，则导入MA_copy，保证不存在异质性

RISK_FREE_RATE = 0 # 无风险利率  

## 读取数据  
需要读取AED数据和市值数据  
市值数据：数据库 or 本地json (测试用)   


In [137]:
ma_df = pl.read_parquet(SAVE_BASE_DIR + '/MA因子.parquet') if not HETER else pl.read_parquet(SAVE_BASE_DIR + '/MA因子_copy.parquet')
ma_df.head()

date,portfolio,MA,return
date,str,f64,f64
2022-11-01,"""002010""",0.811182,-0.069629
2017-03-01,"""600770""",0.607574,-0.1574
2006-11-01,"""002031""",0.649186,0.0552
2018-12-01,"""300215""",0.748944,0.0412
2015-10-01,"""002170""",0.597621,0.16923


In [138]:
# 市值数据从 DB 读取（statics.market_value）
market_value_df = pl.read_database_uri(
    uri=CONNECTION_URL,
    query='''SELECT stkcd AS "Stkcd", trdmnt AS "Trdmnt", msmvosd AS "Msmvosd" FROM statics.market_value''',
    engine=ENGINE,
)


## 处理数据(测试)

In [139]:
market_value_df = market_value_df.rename(
    {
        'Stkcd':'portfolio',
        'Trdmnt':'date',
        'Msmvosd':'msmvosd'
    }
)

market_value_df.head()

portfolio,date,msmvosd
str,date,f64
"""000557""",1997-05-01,963245.03
"""000031""",1997-07-01,1.3565e6
"""600835""",1997-09-01,177336.0
"""900920""",1997-03-01,107198.0
"""000417""",1997-06-01,415800.0


## 桶排序  
按照AED因子对数据进行桶排序，按分位数构建BUCKETS_NUM个组合，组合内部使用市值加权平均计算组合收益，然后绘制各个组合的累计收益。   
同时，还需要构建一个对冲组合，其收益为最后一个组合和第一个组合的差值。     

### 构建桶  
#### 分桶组合
首先进行分桶，获取每个桶的时序收益(series_list)


In [140]:
# 分桶
ma_df = ma_df.with_columns(
    [
        pl.col('MA').quantile(i / BUCKETS_NUM, interpolation='lower').alias(f'q{i}')
        for i in range(1,BUCKETS_NUM)
    ]
)

joined_df = ma_df.join(market_value_df, on=['portfolio','date'], how='left')
joined_df = joined_df.with_columns(
    (pl.col('msmvosd') / pl.col('msmvosd').sum().over('date')).alias('weight')
)

series_list = list[pl.DataFrame]() # 每一个桶对应的series，date-weighted_sum_ret
mean_ret_list = []  # 每一个桶的月平均收益  
for i in range(0,BUCKETS_NUM): # 按照MA由小到大排序  
    # 第1个组合
    if i == 0:
        bucket_df = joined_df.filter(pl.col(f'MA') <= pl.col(f'q1'))
    elif i == (BUCKETS_NUM - 1):
        bucket_df = joined_df.filter(pl.col(f'MA') > pl.col(f'q{i}'))
    else:
        bucket_df = joined_df.filter((pl.col(f'MA') > pl.col(f'q{i}')) & (pl.col(f'MA') <= pl.col(f'q{i+1}')))
    bucket_df = bucket_df.select(['date','portfolio','weight','return'])
    
    # 获取series
    series_df = bucket_df.group_by(['date']).agg(
        (pl.col('weight') * pl.col('return')).sum().alias('weighted_sum_ret'),
        pl.lit(i).alias('bucket_id')
    )

    # 添加到series_list
    series_list.append(series_df)

In [141]:
# 合并
combined_series = pl.concat(series_list, how='vertical').sort(['date','bucket_id'])
combined_series = combined_series.with_columns(pl.col('bucket_id').cast(pl.Utf8)) # 转为字符串，便于添加对冲组合

#### 对冲组合  
做多第BUCKET_NUM-1个组合，做空第1个组合。 

In [142]:
# 构建对冲组合
low_bucket = combined_series.filter(pl.col('bucket_id') == str(0)).rename({'weighted_sum_ret':'low_bucket_ret'})
high_bucket = combined_series.filter(pl.col('bucket_id') == str(BUCKETS_NUM - 1)).rename({'weighted_sum_ret':'high_bucket_ret'})  
joined_bucket = low_bucket.join(high_bucket, on='date', how='left')
joined_bucket = joined_bucket.with_columns(
    pl.col('high_bucket_ret').fill_null(0).alias('high_bucket_ret'),
    pl.col('low_bucket_ret').fill_null(0).alias('low_bucket_ret'),
)
joined_bucket = joined_bucket.with_columns(
    (pl.col('high_bucket_ret') - pl.col('low_bucket_ret')).alias('weighted_sum_ret')
)
joined_bucket = joined_bucket.select('date','weighted_sum_ret').with_columns(
    pl.lit('对冲组合').alias('bucket_id')
)

joined_bucket.head()

date,weighted_sum_ret,bucket_id
date,f64,str
2004-12-01,0.001063,"""对冲组合"""
2005-01-01,-0.001534,"""对冲组合"""
2005-02-01,0.005351,"""对冲组合"""
2005-03-01,0.005163,"""对冲组合"""
2005-04-01,0.0137,"""对冲组合"""


In [143]:
combined_series = pl.concat([combined_series, joined_bucket], how='vertical').sort(['date','bucket_id'])
combined_series.head()

date,weighted_sum_ret,bucket_id
date,f64,str
2004-12-01,-0.004801,"""0"""
2004-12-01,-0.009872,"""1"""
2004-12-01,-0.010538,"""2"""
2004-12-01,-0.014335,"""3"""
2004-12-01,-0.003737,"""4"""


In [144]:
if SAVE:
    combined_series.write_parquet(SAVE_BASE_DIR + '/分桶组合收益.parquet')

## 表现

### 样本数 
统计每个桶的样本数  

In [145]:
count = combined_series.group_by('bucket_id').agg(pl.col('weighted_sum_ret').count().alias('count'))
count.sort('bucket_id')


bucket_id,count
str,u32
"""0""",241
"""1""",241
"""2""",241
"""3""",241
"""4""",241
"""对冲组合""",241


### 平均收益
#### 每个桶的平均收益（包括对冲组合）    
对于每个桶序列，计算其各个时期的平均收益, HAC-t 和 p  

计算方式为，使用`weighted_sum_ret`回归常数项，使用HAC-t和p。  

In [146]:
# 每个 bucket：weighted_sum_ret 的 mean、HAC-t、p（只回归常数项，无 market_ret）
def regress_one(s: pl.DataFrame) -> pl.DataFrame:
    g = s.to_pandas()
    bid = g['bucket_id'].iloc[0]
    try:
        y = g['weighted_sum_ret'].to_numpy()
        x = np.ones((len(y), 1))
        results = sm.OLS(y, x).fit(cov_type='HAC', cov_kwds={'maxlags': 4})
        return pl.DataFrame({
            'bucket_id': [bid],
            'mean_return': [float(results.params[0])],
            't': [float(results.tvalues[0])],
            'p': [float(results.pvalues[0])],
        })
    except Exception as e:
        warnings.warn(f'计算{bid}时发生错误: {e}')
        return pl.DataFrame({
            'bucket_id': [bid],
            'mean_return': [0.0],
            't': [0.0],
            'p': [1.0],
        })
mean_tp_table = combined_series.group_by('bucket_id').map_groups(regress_one)

In [147]:
mean_tp_table

bucket_id,mean_return,t,p
str,f64,f64,f64
"""0""",-0.002942,-1.941408,0.052209
"""对冲组合""",0.005893,4.91294,8.9721e-7
"""3""",0.002099,2.212555,0.026928
"""2""",0.000966,0.852555,0.393906
"""1""",-0.000035,-0.025172,0.979917
"""4""",0.002951,4.094102,0.000042


In [148]:
# 格式化：4 位有效数字
mean_tp_table = mean_tp_table.with_columns(
    pl.col('mean_return').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('mean_return'),
    pl.col('t').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('t'),
    pl.col('p').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('p'),
)
# t、p 加括号
mean_tp_table = mean_tp_table.select(
    pl.col('bucket_id'),
    pl.col('mean_return'),
    (pl.lit('[') + pl.col('t') + pl.lit(']')).alias('t'),
    (pl.lit('(') + pl.col('p') + pl.lit(')')).alias('p'),
)
# 居中对齐到 12 位
mean_tp_table = mean_tp_table.with_columns(
    pl.col('mean_return').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('mean_return'),
    pl.col('t').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('t'),
    pl.col('p').map_elements(lambda s: str(s).center(12), return_dtype=pl.Utf8).alias('p'),
)
# 合并为一行展示
mean_tp_table = mean_tp_table.select(
    pl.col('bucket_id'),
    (pl.col('mean_return') + pl.lit('\n') + pl.col('t') + pl.lit('\n') + pl.col('p')).alias('mean_return'),
).sort('bucket_id')

with pl.Config(tbl_width_chars=200, tbl_cols=20, fmt_str_lengths=200):
    display(mean_tp_table)

bucket_id,mean_return
str,str
"""0""",""" -0.002942 [-1.941] (0.05221) """
"""1""",""" -3.494e-05 [-0.02517] (0.9799) """
"""2""",""" 0.000966 [0.8526] (0.3939) """
"""3""",""" 0.002099 [2.213] (0.02693) """
"""4""",""" 0.002951 [4.094] (4.238e-05) """
"""对冲组合""",""" 0.005893 [4.913] (8.972e-07) """


将对冲组合的收益序列添加到桶排序的底部  

绘制panel_list的累计收益曲线  

绘制方法为，对于每一个时序数据，计算每一期累计收益，然后绘制成曲线。  

In [149]:
# 累加
combined_series = combined_series.with_columns(pl.col('weighted_sum_ret').cum_sum().over('bucket_id').alias('cum_return'))

# 百分化
combined_series = combined_series.with_columns(
    pl.col('cum_return').mul(100).alias('cum_return')
)

# 平滑
ROLLING_WINDOW = 4
MIN_SAMPLES = 1
combined_series = combined_series.with_columns(
    pl.col('cum_return').rolling_mean(window_size=ROLLING_WINDOW, min_samples=MIN_SAMPLES).over('bucket_id').alias('cum_return_smooth')
)

# 绘图
fig = px.line(combined_series, x='date', y='cum_return_smooth', color='bucket_id')
fig.update_layout(
    title=f'累计收益按桶分组(平滑窗口={ROLLING_WINDOW},最小样本={MIN_SAMPLES})',           # 图标题
    xaxis_title='日期',                # x 轴名称
    yaxis_title=f'累计收益率(%)',           # y 轴名称
)
fig.show()
    
    

In [150]:
if SAVE:
    fig.write_image(SAVE_BASE_DIR + '/基准回归-累计收益率分桶图.png')

### Sharp  
计算每个组合的Sharp比率    

计算方式为：$(mean(ret) - RISK\_FREE\_RATE) / std(ret)$


In [151]:
sharp_series = combined_series.group_by(['bucket_id']).agg(
    pl.col('weighted_sum_ret').mean().alias('mean_ret'),
    pl.col('weighted_sum_ret').std().alias('std_ret'),
)

sharp_series = sharp_series.select(
    pl.col('bucket_id'),
    ((pl.col('mean_ret') - RISK_FREE_RATE) / pl.col('std_ret')).alias('sharp')
)

sharp_series = sharp_series.with_columns(
    pl.col('sharp').map_elements(lambda x: format(float(x), '.4g'), return_dtype=pl.Utf8).alias('sharp'),
)
sharp_series = sharp_series.sort('bucket_id')
sharp_series

bucket_id,sharp
str,str
"""0""","""-0.1299"""
"""1""","""-0.001947"""
"""2""","""0.06658"""
"""3""","""0.175"""
"""4""","""0.3244"""
"""对冲组合""","""0.3312"""


### 合并+保存 收益和sharp

In [152]:
performance = count.join(mean_tp_table, on='bucket_id', how='left').join(sharp_series, on='bucket_id', how='left').sort('bucket_id')

if SAVE:
    performance.write_parquet(SAVE_BASE_DIR + '/分桶表现.parquet')

In [153]:
with pl.Config(tbl_width_chars=200, tbl_cols=20, fmt_str_lengths=200):
    display(performance)

bucket_id,count,mean_return,sharp
str,u32,str,str
"""0""",241,""" -0.002942 [-1.941] (0.05221) ""","""-0.1299"""
"""1""",241,""" -3.494e-05 [-0.02517] (0.9799) ""","""-0.001947"""
"""2""",241,""" 0.000966 [0.8526] (0.3939) ""","""0.06658"""
"""3""",241,""" 0.002099 [2.213] (0.02693) ""","""0.175"""
"""4""",241,""" 0.002951 [4.094] (4.238e-05) ""","""0.3244"""
"""对冲组合""",241,""" 0.005893 [4.913] (8.972e-07) ""","""0.3312"""
